In [ ]:
from docling.document_converter import DocumentConverter
from langchain_core.documents import Document

converter = DocumentConverter()
result = converter.convert("CELEX_02013R0575-20250629_EN_TXT.pdf")
doc = result.document


In [ ]:
# Debug: Let's see what we're actually dealing with
print("=== DOCLING DOCUMENT ANALYSIS ===")

# Check what items exist and their properties
headings_found = []
articles_found = []
all_text_samples = []

count = 0
for item, level in doc.iterate_items():
    count += 1
    
    # Get text content from the item
    txt = ""
    if hasattr(item, 'text') and item.text:
        txt = item.text.strip()
    elif hasattr(item, 'caption') and item.caption:
        txt = item.caption.strip()
    elif hasattr(item, 'title') and item.title:
        txt = item.title.strip()
    
    # Check if it has a label
    if hasattr(item, 'label'):
        print(f"Item {count}: Type={type(item).__name__}, Label='{item.label}', Text='{txt[:100]}...'")
        
        # Collect headings
        if item.label == "heading":
            headings_found.append(txt)
            
        # Check for articles specifically
        if txt and ("article" in txt.lower() or "Article" in txt):
            articles_found.append(f"Label: {item.label}, Text: {txt}")
    
    # Collect some text samples
    if txt and len(all_text_samples) < 20:
        all_text_samples.append(f"Type: {type(item).__name__}, Label: {getattr(item, 'label', 'None')}, Text: {txt[:100]}")
    
    if count > 100:  # Limit output
        break

print(f"\n=== SUMMARY ===")
print(f"Total items processed: {count}")
print(f"Headings found: {len(headings_found)}")
print(f"Items mentioning 'Article': {len(articles_found)}")

print(f"\n=== HEADINGS FOUND ===")
for i, heading in enumerate(headings_found[:10]):  # Show first 10
    print(f"{i+1}: {heading}")

print(f"\n=== ARTICLE MENTIONS ===")
for article in articles_found[:10]:  # Show first 10
    print(article)

print(f"\n=== SAMPLE TEXT CONTENT ===")
for sample in all_text_samples[:10]:  # Show first 10
    print(sample)

In [ ]:
# Look for Article patterns in any text
chunks = []
current = None

print("=== TRYING FLEXIBLE ARTICLE EXTRACTION ===")

for item, level in doc.iterate_items():
    # Get text content
    txt = ""
    if hasattr(item, 'text') and item.text:
        txt = item.text.strip()
    elif hasattr(item, 'caption') and item.caption:
        txt = item.caption.strip()
    elif hasattr(item, 'title') and item.title:
        txt = item.title.strip()
    
    if not txt:
        continue
    
    # Look for Article patterns (more flexible)
    import re
    article_match = re.match(r'^Article\s+(\d+)', txt)
    
    if article_match:
        print(f"Found article: {txt[:100]}")
        
        # Save previous chunk
        if current:
            chunks.append(Document(**current))
        
        article_num = article_match.group(1)
        
        # Get page number
        try:
            page_num = item.prov[0].page_no if hasattr(item, 'prov') and item.prov else 1
        except:
            page_num = 1
        
        current = {
            "page_content": txt + "\n",
            "metadata": {
                "type": "article",
                "article_no": f"Article {article_num}",
                "page": page_num,
                "item_type": type(item).__name__,
                "label": getattr(item, 'label', 'unknown')
            }
        }
    elif current:
        current["page_content"] += txt + "\n"

# Add final chunk
if current:
    chunks.append(Document(**current))

print(f"\n=== RESULTS ===")
print(f"Extracted {len(chunks)} chunks using flexible approach")
for i, chunk in enumerate(chunks[:10]):
    print(f"\nChunk {i+1}:")
    print(f"Metadata: {chunk.metadata}")
    print(f"Content preview: {chunk.page_content[:200]}...")

In [ ]:

import os
from dotenv import load_dotenv
from langchain_astradb import AstraDBVectorStore
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

# Load environment variables
load_dotenv()

print(f"✅ Successfully extracted {len(chunks)} legal document chunks with Docling!")
print(f"Sample chunk metadata: {chunks[0].metadata}")
print(f"Sample content preview: {chunks[0].page_content[:200]}...")

✅ Successfully extracted 737 legal document chunks with Docling!
Sample chunk metadata: {'type': 'article', 'article_no': 'Article 1', 'page': 3, 'item_type': 'SectionHeaderItem', 'label': <DocItemLabel.SECTION_HEADER: 'section_header'>}
Sample content preview: Article  1
Scope
This Regulation lays down uniform rules concerning general prudential requirements  that  institutions,  financial  holding  companies  and  mixed financial  holding  companies  super...


In [ ]:
# Initialize NVIDIA embeddings
print("🔧 Initializing NVIDIA embeddings...")
embeddings = NVIDIAEmbeddings(
    model="nvidia/nv-embedqa-e5-v5",
    api_key=os.getenv("NVIDIA_API_KEY")
)

# Connect to Astra DB
print("🗄️ Connecting to Astra DB...")
collection_name = "legal_docling_chunks"

vectorstore = AstraDBVectorStore(
    embedding=embeddings,
    collection_name=collection_name,
    token=os.getenv("ASTRA_DB_TOKEN"),
    api_endpoint=os.getenv("ASTRA_DB_API_ENDPOINT"),
)

print(f"✅ Connected to Astra DB collection: {collection_name}")

In [ ]:
# Fix: Split large chunks to fit NVIDIA's 512 token limit
from langchain_text_splitters import RecursiveCharacterTextSplitter

def estimate_tokens(text):
    """Rough estimation: 1 token ≈ 4 characters"""
    return len(text) / 4

def split_large_chunks(chunks, max_tokens=512):
    """Split chunks that exceed the token limit"""
    print(f"🔧 Checking chunks for NVIDIA's {max_tokens} token limit...")
    
    valid_chunks = []
    oversized_count = 0
    
    # Text splitter for oversized chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1800,  # ~450 tokens (safe margin)
        chunk_overlap=200,
        separators=[
            "\n\nArticle ",   # Split at article boundaries
            "\n\n",          # Paragraph breaks
            "\n",            # Line breaks
            ". ",            # Sentence breaks
            " ",             # Word breaks
            ""
        ],
        keep_separator=True,
        length_function=len,
    )
    
    for i, chunk in enumerate(chunks):
        estimated_tokens = estimate_tokens(chunk.page_content)
        
        if estimated_tokens <= max_tokens:
            # Chunk is fine as-is
            valid_chunks.append(chunk)
        else:
            # Chunk is too large, split it
            oversized_count += 1
            print(f"  Splitting chunk {i+1} ({estimated_tokens:.0f} tokens)")
            
            # Split the oversized chunk
            sub_chunks = text_splitter.split_documents([chunk])
            
            # Add metadata to sub-chunks
            for j, sub_chunk in enumerate(sub_chunks):
                # Preserve original metadata and add sub-chunk info
                sub_chunk.metadata = chunk.metadata.copy()
                sub_chunk.metadata['sub_chunk'] = j + 1
                sub_chunk.metadata['total_sub_chunks'] = len(sub_chunks)
                sub_chunk.metadata['original_chunk_tokens'] = int(estimated_tokens)
                
                # Verify sub-chunk size
                sub_tokens = estimate_tokens(sub_chunk.page_content)
                if sub_tokens <= max_tokens:
                    valid_chunks.append(sub_chunk)
                else:
                    print(f"    Warning: Sub-chunk still too large ({sub_tokens:.0f} tokens)")
    
    print(f"✅ Processed {len(chunks)} chunks:")
    print(f"   - {len(chunks) - oversized_count} chunks were within limit")
    print(f"   - {oversized_count} chunks were split")
    print(f"   - Final total: {len(valid_chunks)} chunks")
    
    return valid_chunks

# Split the chunks
print("Preparing chunks for NVIDIA embeddings...")
valid_chunks = split_large_chunks(chunks, max_tokens=512)

In [ ]:
# Now add the properly sized chunks to Astra DB
print(f"📚 Adding {len(valid_chunks)} properly-sized chunks to Astra DB...")

# Process in smaller batches for stability
batch_size = 25  # Smaller batches
total_batches = (len(valid_chunks) + batch_size - 1) // batch_size

successfully_added = 0
failed_chunks = []

for i in range(0, len(valid_chunks), batch_size):
    batch = valid_chunks[i:i + batch_size]
    batch_num = i // batch_size + 1
    
    print(f"Processing batch {batch_num}/{total_batches} ({len(batch)} chunks)...")
    
    try:
        # Check batch for token limits before adding
        for chunk in batch:
            tokens = estimate_tokens(chunk.page_content)
            if tokens > 512:
                print(f"  Warning: Chunk still has {tokens:.0f} tokens")
        
        vectorstore.add_documents(batch)
        successfully_added += len(batch)
        print(f"  ✅ Batch {batch_num} added successfully")
        
    except Exception as e:
        print(f"  ❌ Batch {batch_num} failed: {str(e)[:100]}")
        failed_chunks.extend(batch)

print(f"\n📊 Results:")
print(f"✅ Successfully added: {successfully_added} chunks")
print(f"❌ Failed: {len(failed_chunks)} chunks")

if failed_chunks:
    print(f"\nDebugging first failed chunk:")
    failed_chunk = failed_chunks[0]
    print(f"Content length: {len(failed_chunk.page_content)}")
    print(f"Estimated tokens: {estimate_tokens(failed_chunk.page_content):.0f}")
    print(f"Content preview: {failed_chunk.page_content[:200]}...")

In [42]:
# Save valid_chunks as pickle file
import pickle
from datetime import datetime

# Create filename with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
pickle_filename = f"legal_chunks_docling_{timestamp}.pkl"

print(f"💾 Saving {len(valid_chunks)} chunks to pickle file...")

# Save the chunks
with open(pickle_filename, 'wb') as f:
    pickle.dump(valid_chunks, f)

print(f"✅ Chunks saved to: {pickle_filename}")
print(f"📊 File contains {len(valid_chunks)} processed legal document chunks")

# Also save metadata summary
metadata_summary = {
    'total_chunks': len(valid_chunks),
    'source_document': 'CELEX_02013R0575-20250629_EN_TXT.pdf',
    'processing_date': datetime.now().isoformat(),
    'chunk_types': [chunk.metadata.get('type', 'unknown') for chunk in valid_chunks],
    'articles_found': list(set([chunk.metadata.get('article_no', 'Unknown') for chunk in valid_chunks])),
    'processing_method': 'docling_with_nvidia_token_splitting'
}

summary_filename = f"legal_chunks_summary_{timestamp}.pkl"
with open(summary_filename, 'wb') as f:
    pickle.dump(metadata_summary, f)

print(f"📋 Metadata summary saved to: {summary_filename}")

💾 Saving 1618 chunks to pickle file...
✅ Chunks saved to: legal_chunks_docling_20251118_095401.pkl
📊 File contains 1618 processed legal document chunks
📋 Metadata summary saved to: legal_chunks_summary_20251118_095401.pkl


In [36]:
# Complete Legal RAG System - Add this to your notebook

# Initialize Google Gemini for legal analysis
print("🤖 Initializing Google Gemini for legal document analysis...")
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1,  # Low temperature for accuracy in legal context
    google_api_key=os.getenv("GEMINI_API_KEY")
)

# Create legal-specific prompt template
legal_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a specialized legal document assistant with expertise in financial regulations, particularly the Capital Requirements Regulation (CRR). 

Your role is to:
1. Provide accurate, precise answers based solely on the provided legal document context
2. Always cite specific articles, sections, or provisions when referencing information
3. Distinguish between mandatory requirements ("shall", "must") and optional provisions ("may", "should")
4. Explain complex legal concepts in clear, professional language
5. When uncertain, clearly state limitations and suggest consulting legal counsel

Important guidelines:
- Only use information from the provided context
- Never speculate or provide general legal advice
- Always reference specific article numbers when applicable
- Maintain professional, formal tone appropriate for legal documentation"""),
    
    ("user", """Based on the following legal document excerpts, please answer the question:

Context: {context}

Question: {question}

Please provide a comprehensive answer with specific references to articles and provisions.""")
])

# Create the legal RAG chain
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    """Format retrieved documents for the prompt"""
    formatted = []
    for doc in docs:
        article_info = doc.metadata.get('article_no', 'Unknown Article')
        page_info = doc.metadata.get('page', 'Unknown Page')
        content = doc.page_content.strip()
        formatted.append(f"[{article_info}, Page {page_info}]\n{content}")
    return "\n\n---\n\n".join(formatted)

# Create retriever
print("🔍 Setting up legal document retriever...")
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 8}  # Retrieve top 8 most relevant chunks
)

# Build the complete RAG chain
legal_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | legal_prompt
    | llm
    | StrOutputParser()
)

print("✅ Legal RAG system is ready!")
print("\n🎯 System Summary:")
print(f"📄 Document: CELEX_02013R0575-20250629_EN_TXT.pdf (Capital Requirements Regulation)")
print(f"📚 Chunks stored: {successfully_added} legal document sections")
print(f"🧠 Embeddings: NVIDIA nv-embedqa-e5-v5")
print(f"🗄️ Vector Store: Astra DB")
print(f"🤖 LLM: Google Gemini 2.0 Flash")
print(f"📖 Specialized for: Financial regulation legal analysis")

🤖 Initializing Google Gemini for legal document analysis...
🔍 Setting up legal document retriever...
✅ Legal RAG system is ready!

🎯 System Summary:
📄 Document: CELEX_02013R0575-20250629_EN_TXT.pdf (Capital Requirements Regulation)
📚 Chunks stored: 1618 legal document sections
🧠 Embeddings: NVIDIA nv-embedqa-e5-v5
🗄️ Vector Store: Astra DB
🤖 LLM: Google Gemini 2.0 Flash
📖 Specialized for: Financial regulation legal analysis


In [37]:
# Test your legal RAG system
def query_legal_document(question):
    """Query the legal RAG system"""
    try:
        print(f"🔍 Searching for: {question}")
        print("=" * 50)
        
        # Get answer from RAG chain
        response = legal_rag_chain.invoke(question)
        
        print("📋 Legal Analysis:")
        print(response)
        print("=" * 50)
        
        return response
    except Exception as e:
        print(f"❌ Error querying system: {e}")
        return None

# Example legal queries you can try:
sample_queries = [
    "What are the capital requirements for credit institutions under Article 92?",
    "What is the definition of Common Equity Tier 1 capital?",
    "What are the requirements for large exposures?",
    "How are credit risk adjustments calculated?",
    "What are the liquidity coverage requirements?"
]

print("🎯 Sample Legal Queries:")
for i, query in enumerate(sample_queries, 1):
    print(f"{i}. {query}")

print("\n💡 Try querying your system:")
print("response = query_legal_document('Your question about the regulation')")

🎯 Sample Legal Queries:
1. What are the capital requirements for credit institutions under Article 92?
2. What is the definition of Common Equity Tier 1 capital?
3. What are the requirements for large exposures?
4. How are credit risk adjustments calculated?
5. What are the liquidity coverage requirements?

💡 Try querying your system:
response = query_legal_document('Your question about the regulation')


In [41]:
query_legal_document("What are the requirements for large exposures?") 

🔍 Searching for: What are the requirements for large exposures?


2025-11-17 11:54:07,243 - INFO - cursor fetching a page: (empty page state) from legal_docling_chunks
2025-11-17 11:54:08,475 - INFO - HTTP Request: POST https://50cf877b-c447-42e7-b570-8e1ca0a0dbde-us-east1.apps.astra.datastax.com/api/json/v1/default_keyspace/legal_docling_chunks "HTTP/1.1 200 OK"
2025-11-17 11:54:08,478 - INFO - cursor finished fetching a page: (empty page state) from legal_docling_chunks


📋 Legal Analysis:
Based on the provided legal document excerpts, the requirements for large exposures are as follows:

1.  **Definition of a Large Exposure**
    An institution's exposure to a client or a group of connected clients is considered a large exposure if its value is equal to or exceeds 10% of the institution's Tier 1 capital (Article 392).

2.  **General Monitoring and Control**
    Institutions **shall** monitor and control their large exposures in accordance with Part Four of this Regulation (Article 387).

3.  **Administrative and Accounting Procedures**
    An institution **shall** have sound administrative and accounting procedures and adequate internal control mechanisms for identifying, managing, monitoring, reporting, and recording all large exposures and subsequent changes to them (Article 393).

4.  **Limits to Large Exposures**
    *   An institution **shall not** incur an exposure to a client or group of connected clients that exceeds 25% of its Tier 1 capital, 

"Based on the provided legal document excerpts, the requirements for large exposures are as follows:\n\n1.  **Definition of a Large Exposure**\n    An institution's exposure to a client or a group of connected clients is considered a large exposure if its value is equal to or exceeds 10% of the institution's Tier 1 capital (Article 392).\n\n2.  **General Monitoring and Control**\n    Institutions **shall** monitor and control their large exposures in accordance with Part Four of this Regulation (Article 387).\n\n3.  **Administrative and Accounting Procedures**\n    An institution **shall** have sound administrative and accounting procedures and adequate internal control mechanisms for identifying, managing, monitoring, reporting, and recording all large exposures and subsequent changes to them (Article 393).\n\n4.  **Limits to Large Exposures**\n    *   An institution **shall not** incur an exposure to a client or group of connected clients that exceeds 25% of its Tier 1 capital, after